# 01 — Exploratory Data Analysis
**Olist E-Commerce Dataset** | 9 tablas relacionales | ~100k órdenes

Objetivo: entender la estructura, calidad y distribuciones de cada tabla antes de limpiar.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 4)

In [ ]:
RAW = '../data/raw/'

orders       = pd.read_csv(RAW + 'olist_orders_dataset.csv')
order_items  = pd.read_csv(RAW + 'olist_order_items_dataset.csv')
payments     = pd.read_csv(RAW + 'olist_order_payments_dataset.csv')
reviews      = pd.read_csv(RAW + 'olist_order_reviews_dataset.csv')
customers    = pd.read_csv(RAW + 'olist_customers_dataset.csv')
products     = pd.read_csv(RAW + 'olist_products_dataset.csv')
sellers      = pd.read_csv(RAW + 'olist_sellers_dataset.csv')
geolocation  = pd.read_csv(RAW + 'olist_geolocation_dataset.csv')
translations = pd.read_csv(RAW + 'product_category_name_translation.csv')

tables = {
    'orders': orders, 'order_items': order_items, 'payments': payments,
    'reviews': reviews, 'customers': customers, 'products': products,
    'sellers': sellers, 'geolocation': geolocation, 'translations': translations
}

print('=== Dataset overview ===')
for name, df in tables.items():
    print(f'{name:20s} {str(df.shape):15s} nulls: {df.isnull().sum().sum()}')

---
## 1. Orders

In [ ]:
print(orders.dtypes)
print('\nNulls:')
print(orders.isnull().sum())

In [ ]:
orders.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Order status distribution
orders['order_status'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Order Status Distribution')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

# Orders over time
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['year_month'] = orders['order_purchase_timestamp'].dt.to_period('M')
monthly = orders.groupby('year_month').size()
monthly.plot(ax=axes[1], color='steelblue', marker='o', markersize=3)
axes[1].set_title('Orders per Month')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

print(f'Date range: {orders["order_purchase_timestamp"].min().date()} → {orders["order_purchase_timestamp"].max().date()}')
print(f'Delivered: {(orders["order_status"] == "delivered").mean():.1%}')

---
## 2. Order Items

In [ ]:
print(order_items.dtypes)
print('\nNulls:')
print(order_items.isnull().sum())
print('\nStats:')
order_items[['price', 'freight_value']].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Price distribution (capped at 99th pct)
price_cap = order_items['price'].quantile(0.99)
order_items[order_items['price'] <= price_cap]['price'].hist(bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Price Distribution (cap p99)')

# Freight distribution
freight_cap = order_items['freight_value'].quantile(0.99)
order_items[order_items['freight_value'] <= freight_cap]['freight_value'].hist(bins=50, ax=axes[1], color='coral')
axes[1].set_title('Freight Distribution (cap p99)')

# Items per order
items_per_order = order_items.groupby('order_id')['order_item_id'].max()
items_per_order.value_counts().sort_index().head(10).plot(kind='bar', ax=axes[2], color='mediumseagreen')
axes[2].set_title('Items per Order')
axes[2].set_xlabel('# items')

plt.tight_layout()
plt.show()

print(f'Total revenue (price): R$ {order_items["price"].sum():,.0f}')
print(f'Total freight: R$ {order_items["freight_value"].sum():,.0f}')
print(f'Avg price: R$ {order_items["price"].mean():.2f} | Median: R$ {order_items["price"].median():.2f}')
print(f'Orders with >1 item: {(items_per_order > 1).mean():.1%}')

---
## 3. Payments

In [ ]:
print(payments.dtypes)
print('\nNulls:')
print(payments.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Payment type
payments['payment_type'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Payment Type Distribution')
axes[0].tick_params(axis='x', rotation=30)

# Installments
payments['payment_installments'].value_counts().sort_index().head(12).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Payment Installments Distribution')
axes[1].set_xlabel('# installments')

plt.tight_layout()
plt.show()

print(payments['payment_type'].value_counts())
print(f'\nAvg payment value: R$ {payments["payment_value"].mean():.2f}')

---
## 4. Reviews

In [ ]:
print(reviews.dtypes)
print('\nNulls:')
print(reviews.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Score distribution
reviews['review_score'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Review Score Distribution')
axes[0].set_xlabel('Score')

# % with comment
has_comment = reviews['review_comment_message'].notna()
pd.Series({'With comment': has_comment.sum(), 'No comment': (~has_comment).sum()}).plot(
    kind='bar', ax=axes[1], color=['mediumseagreen', 'lightcoral']
)
axes[1].set_title('Reviews With/Without Comment')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(f'Avg review score: {reviews["review_score"].mean():.2f}')
print(f'Score 1-2 (negative): {(reviews["review_score"] <= 2).mean():.1%}')
print(f'Score 4-5 (positive): {(reviews["review_score"] >= 4).mean():.1%}')

---
## 5. Customers

In [ ]:
print(customers.dtypes)
print('\nNulls:')
print(customers.isnull().sum())
print(f'\nTotal customers: {len(customers):,}')
print(f'Unique customers (customer_unique_id): {customers["customer_unique_id"].nunique():,}')
print('-> Diferencia = clientes que hicieron más de una orden')

In [ ]:
top_states = customers['customer_state'].value_counts().head(10)
top_states.plot(kind='bar', color='steelblue', figsize=(12, 4))
plt.title('Top 10 States by Number of Customers')
plt.xlabel('')
plt.tight_layout()
plt.show()

print(top_states)

---
## 6. Products

In [ ]:
print(products.dtypes)
print('\nNulls:')
print(products.isnull().sum())

In [ ]:
# Top categories (with English translation)
products_translated = products.merge(translations, on='product_category_name', how='left')
top_cats = products_translated['product_category_name_english'].value_counts().head(15)

top_cats.plot(kind='barh', color='steelblue', figsize=(12, 5))
plt.title('Top 15 Product Categories')
plt.xlabel('# products')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f'Total categories: {products["product_category_name"].nunique()}')
print(f'Products without category: {products["product_category_name"].isna().sum()}')

---
## 7. Sellers

In [ ]:
print(sellers.dtypes)
print('\nNulls:')
print(sellers.isnull().sum())
print(f'\nTotal sellers: {len(sellers):,}')

In [ ]:
sellers['seller_state'].value_counts().head(10).plot(kind='bar', color='steelblue', figsize=(12, 4))
plt.title('Top 10 States by Number of Sellers')
plt.xlabel('')
plt.tight_layout()
plt.show()

---
## 8. Geolocation

In [ ]:
print(geolocation.dtypes)
print('\nNulls:')
print(geolocation.isnull().sum())
print(f'\nTotal rows: {len(geolocation):,}')
print(f'Unique zip codes: {geolocation["geolocation_zip_code_prefix"].nunique():,}')
print('-> Múltiples coords por zip code — necesita deduplicar en cleaning')

---
## 9. Category Translations

In [ ]:
print(f'Total categories with translation: {len(translations)}')
print(f'Categories in products table: {products["product_category_name"].nunique()}')
print(f'Categories WITHOUT translation: {products["product_category_name"].nunique() - len(translations)}')
translations.head(5)

---
## 10. Delivery Time Analysis

In [ ]:
delivered = orders[orders['order_status'] == 'delivered'].copy()

for col in ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    delivered[col] = pd.to_datetime(delivered[col])

delivered['delivery_days'] = (delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']).dt.days
delivered['estimated_days'] = (delivered['order_estimated_delivery_date'] - delivered['order_purchase_timestamp']).dt.days
delivered['late'] = delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Delivery days distribution
delivered['delivery_days'].clip(0, 60).hist(bins=40, ax=axes[0], color='steelblue')
axes[0].axvline(delivered['delivery_days'].median(), color='red', linestyle='--', label=f'Median: {delivered["delivery_days"].median():.0f}d')
axes[0].set_title('Actual Delivery Time (days)')
axes[0].legend()

# Late vs on-time
delivered['late'].value_counts().rename({True: 'Late', False: 'On time'}).plot(
    kind='bar', ax=axes[1], color=['lightcoral', 'mediumseagreen']
)
axes[1].set_title('Late vs On-Time Deliveries')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(f'Avg delivery time: {delivered["delivery_days"].mean():.1f} days')
print(f'Median delivery time: {delivered["delivery_days"].median():.0f} days')
print(f'Late orders: {delivered["late"].mean():.1%}')

---
## 11. Schema — Relaciones entre tablas

```
customers ──── orders ──── order_items ──── products ──── translations
                  │             │
                  │         sellers
                  │
               payments
               reviews

geolocation ←── (zip_code) ─── customers / sellers
```

**Join keys:**
- `orders.customer_id` → `customers.customer_id`
- `order_items.order_id` → `orders.order_id`
- `order_items.product_id` → `products.product_id`
- `order_items.seller_id` → `sellers.seller_id`
- `payments.order_id` → `orders.order_id`
- `reviews.order_id` → `orders.order_id`
- `products.product_category_name` → `translations.product_category_name`

In [ ]:
# Verificar integridad referencial
print('=== Referential integrity check ===')

# order_items → orders
orphan_items = ~order_items['order_id'].isin(orders['order_id'])
print(f'order_items sin orden padre: {orphan_items.sum()}')

# payments → orders
orphan_payments = ~payments['order_id'].isin(orders['order_id'])
print(f'payments sin orden padre: {orphan_payments.sum()}')

# reviews → orders
orphan_reviews = ~reviews['order_id'].isin(orders['order_id'])
print(f'reviews sin orden padre: {orphan_reviews.sum()}')

# order_items → products
orphan_products = ~order_items['product_id'].isin(products['product_id'])
print(f'order_items sin producto: {orphan_products.sum()}')

# order_items → sellers
orphan_sellers = ~order_items['seller_id'].isin(sellers['seller_id'])
print(f'order_items sin seller: {orphan_sellers.sum()}')

---
## Resumen — Issues para Cleaning

Anotar acá los hallazgos que requieren acción en `02_cleaning.ipynb`.

In [ ]:
print("""
ISSUES IDENTIFICADOS — con números reales
==========================================

orders (99,441 filas):
  - 4 columnas de timestamps son string -> convertir a datetime
  - Nulls esperados: order_approved_at (160), order_delivered_carrier_date (1,783),
    order_delivered_customer_date (2,965) — órdenes canceladas o en tránsito
  - 97% son status='delivered' -> filtrar solo esas para análisis principal
  - 775 órdenes sin items en order_items -> canceladas antes de procesar, descartar
  - Rango de fechas: sep 2016 - oct 2018

order_items (112,650 filas):
  - shipping_limit_date es string -> convertir a datetime
  - Precio mínimo de R$ 0.85 -> revisar si son registros válidos
  - 90% de órdenes tienen 1 solo ítem (comportamiento normal del marketplace)

payments (103,886 filas):
  - 3 registros con payment_type='not_defined' -> eliminar
  - 2 registros con payment_installments=0 -> revisar/eliminar

reviews (99,224 filas):
  - review_comment_message: 58,247 nulls (58.6%) — ESPERADO, comentario es opcional
  - review_comment_title: 87,656 nulls (88.3%) — ESPERADO, casi nadie lo usa
  - No requieren acción, son nulls legítimos

customers (99,441 filas):
  - customer_id != customer_unique_id: 3,345 clientes hicieron más de 1 orden
  - Siempre usar customer_unique_id para contar clientes reales
  - Tasa de recompra muy baja (~3.5%)

products (32,951 filas):
  - product_name_lenght / product_description_lenght: typo en nombre de columna (renombrar)
  - 610 productos sin product_category_name -> asignar 'unknown'
  - 2 categorías sin traducción al inglés: 'pc_gamer' y
    'portateis_cozinha_e_preparadores_de_alimentos' -> agregar manualmente

geolocation (1,000,163 filas):
  - 19,015 zip codes únicos pero promedio 52.6 filas por zip (max: 1,146)
  - Deduplicar: quedarse con median(lat) y median(lng) por zip -> de 1M a 19k filas

INTEGRIDAD REFERENCIAL: OK
  - 0 huérfanos en order_items, payments, reviews
  - Los joins entre tablas son limpios
""")